# Top+Side View Volume Estimation
# Feature Extractor with Trainable Last Stage + POV Fusion + Regression Head

This notebook trains and evaluates different combinations of feature extractors with unfrozen last stage, and regression heads on the top+side dataset. Uses concat fusion strategy.


In [ ]:
import datetime
import warnings
from IPython.display import display
from src.training_and_evaluation import *
from src.trainable_pipelines import *

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

## Setup

In [ ]:
LAST_LAYER_OUTER_SPLITS = 3
LAST_LAYER_INNER_SPLITS = 3

LAST_LAYER_BACKBONE_NAMES = [
    "densenet121",
    "vit_b_16"
]

LAST_LAYER_FUSION_NAMES = [
    "concat"
]

LAST_LAYER_REGRESSION_MODEL_CONFIGS = {
    "linear": [
        {
            "lr_backbone": 1e-4,
            "lr_head": 1e-3,
            "weight_decay": 1e-4,
        }
    ],
    "mlp": list(ParameterGrid({
        "hidden_dims": [(64,)],
        "dropout": [0.0],
        "lr_backbone": [1e-4],
        "lr_head": [1e-3, 3e-4],
        "weight_decay": [1e-4, 1e-3],
    }))
}

## Load Data

In [ ]:
samples, image_paths = load_image_paths(CSV_PATH, TOP_FOLDER, SIDE_FOLDER)
print("Samples shape:", samples.shape)
display(samples[["exp_id", "volume", "top_path", "side_path"]].head(8))

## Show Regression Model Configurations

In [ ]:
print("Grid sizes per head:")
for name, grid in LAST_LAYER_REGRESSION_MODEL_CONFIGS.items():
    n = len(grid)
    print(f"  {name:10s}: {n:4d}")

## Nested CV Evaluation (split by experiment IDs)

In [ ]:
all_results = []
nested_artifacts = {}

for backbone_name in LAST_LAYER_BACKBONE_NAMES:
    for fusion_name in LAST_LAYER_FUSION_NAMES:
        print(f"\n{'#'*90}")
        print(f"Evaluating backbone={backbone_name}, fusion={fusion_name}")
        print(f"{'#'*90}")

        image_cache = {}
        mask_cache = {}

        nested_results, oof_predictions, training_histories = run_end_to_end_nested_cv(
            samples_df=samples,
            backbone_name=backbone_name,
            fusion_name=fusion_name,
            head_configs=LAST_LAYER_REGRESSION_MODEL_CONFIGS,
            outer_splits=LAST_LAYER_OUTER_SPLITS,
            inner_splits=LAST_LAYER_INNER_SPLITS,
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            device=DEVICE,
            side_mask_path=SIDE_ROI_MASK,
            top_mask_path1=TOP_ROI_MASK_1,
            top_mask_path2=TOP_ROI_MASK_2,
            patience=PATIENCE,
            min_delta=0.0,
            image_cache=image_cache,
            mask_cache=mask_cache
        )

        summary_df = summarise_nested_results(nested_results, backbone_name, fusion_name)
        all_results.append(summary_df)
        nested_artifacts[(backbone_name, fusion_name)] = {
            "y": samples["volume"].to_numpy(dtype=float),
            "groups": samples["exp_id"].to_numpy(),
            "nested_results": nested_results,
            "oof_predictions": oof_predictions,
            "training_histories": training_histories,
        }

results_df = pd.concat(all_results, ignore_index=True).sort_values(
    ["cv_mae_mean", "cv_rmse_mean", "cv_r2_mean"],
    ascending=[True, True, False],
).reset_index(drop=True)

## Add Dummy Mean Predictions for Comparison

In [ ]:
dummy_pred = np.full(len(samples), samples["volume"].mean(), dtype=float)
dummy_mae = mean_absolute_error(samples["volume"], dummy_pred)
dummy_mse = mean_squared_error(samples["volume"], dummy_pred)
dummy_rmse = np.sqrt(dummy_mse)
dummy_r2 = r2_score(samples["volume"], dummy_pred)

dummy_row = pd.DataFrame([{
    "backbone": "dummy_mean",
    "fusion": "dummy_mean",
    "regressor": "dummy_mean",
    "cv_mae_mean": dummy_mae,
    "cv_mae_std": 0.0,
    "cv_mse_mean": dummy_mse,
    "cv_mse_std": 0.0,
    "cv_rmse_mean": dummy_rmse,
    "cv_rmse_std": 0.0,
    "cv_r2_mean": dummy_r2,
    "cv_r2_std": 0.0,
}])

results_df = pd.concat([results_df, dummy_row], ignore_index=True).sort_values(
    ["cv_mae_mean", "cv_rmse_mean", "cv_r2_mean"],
    ascending=[True, True, False],
).reset_index(drop=True)

## Show Selected Hyperparameters

In [ ]:
for (backbone_name, fusion_name), artifact in nested_artifacts.items():
    print(f"\n{'='*90}")
    print(f"backbone={backbone_name}, fusion={fusion_name}")
    print(f"{'='*90}")
    for head_name, folds in artifact["nested_results"].items():
        has_params = any(f["best_params"] for f in folds)
        if not has_params:
            continue
        print(head_name)
        for f in folds:
            print(f"  Fold {f['fold']}: {f['best_params']}")
        print()


## Show Performance for All Configurations
Prints mean and standard deviation of MAE, MSE, RMSE, and R2 over the CV folds.
Plots out-of-fold predictions for the best configurations.


In [ ]:
display(results_df)

for _, row in results_df.head(5).iterrows():
    key = (row["backbone"], row["fusion"])
    if key in nested_artifacts:
        artifact = nested_artifacts[key]
        y_true = artifact["y"]
        y_pred = artifact["oof_predictions"][row["regressor"]]
        title = f"{row['backbone']} | {row['fusion']} | {row['regressor']}"
        make_oof_plot(y_true, y_pred, title_prefix=title)

## Save Results

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
summary_name = f"top_side_last_stage_unfrozen_{timestamp}.csv"
details_name = f"top_side_last_stage_unfrozen_fold_details_{timestamp}.json"

results_df.to_csv(OUTPUT_DIR / summary_name, index=False)

fold_details = {}
for key, artifact in nested_artifacts.items():
    backbone_name, fusion_name = key
    fold_details[f"{backbone_name}__{fusion_name}"] = artifact["nested_results"]

import json
with open(OUTPUT_DIR / details_name, "w") as f:
    json.dump(fold_details, f, indent=2, default=str)

print("Saved summary:", OUTPUT_DIR / summary_name)
print("Saved fold details:", OUTPUT_DIR / details_name)